# Ordered Logistic Regression Results for Adoption Predictors of Indigenous and Modern Knowledge in Rangeland Management Practices, Northern Kenya: Exploration with `mlcroissant`
This notebook demonstrates how to explore and process the FAIR² dataset using the [`mlcroissant`](https://mlcroissant.ai/) library.

### Dataset Source
The dataset source is a Croissant schema accessible from the following URL:

In [ ]:
# Ensure `mlcroissant` is installed. Run this cell if you haven't already installed it.
!pip install mlcroissant

## 1. Data Loading

Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import numpy as np

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json'

# Load the dataset metadata and inspect its structure
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

# Print dataset name and description
print(f"Dataset Title: {metadata.name}")
print(f"Description: {metadata.description}\n")

## 2. Data Overview

Review available record sets, fields, and their unique `@id` values as defined in the Croissant schema. These IDs enable precise referencing of data entities.

Let's print all record sets and their field `@id`s.

In [ ]:
# List available record sets and fields by their @id
def print_record_sets(dataset):
    record_sets = dataset.record_sets
    if not record_sets:
        print("No record sets found in the dataset metadata.")
        return
    for rs in record_sets:
        print(f"RecordSet name: {rs.name}")
        print(f" - @id: {rs.id}")
        if hasattr(rs, 'fields') and rs.fields:
            print(" - Fields:")
            for field in rs.fields:
                print(f"    - Field name: {field.name}, @id: {field.id}, dataType: {getattr(field, 'data_type', None)}")
        else:
            print("   (No fields found)")
        print()

print_record_sets(dataset)

## 3. Data Extraction

Load data from each available record set into a DataFrame for analysis.

Below we extract each record set by its `@id` as identified in the previous step.

In [ ]:
# Get all record sets IDs
record_sets = dataset.record_sets

record_set_ids = [rs.id for rs in record_sets]
print("Available RecordSet @ids:")
for rid in record_set_ids:
    print(f"  - {rid}")

# Extract all records into dataframes
dataframes = {}
for rid in record_set_ids:
    # Use record_set=rid (the Croissant @id of the RecordSet)
    records = list(dataset.records(record_set=rid))
    dataframes[rid] = pd.DataFrame(records)

# If at least one record set is present, show columns of the first record set
if record_set_ids:
    first_rs = record_set_ids[0]
    print(f"\nFirst RecordSet: {first_rs}")
    df = dataframes[first_rs]
    print("Columns:", df.columns.tolist())
    display(df.head())
else:
    print("No record sets found to extract records.")

## 4. Exploratory Data Analysis (EDA)

Apply EDA steps: filtering, normalization, and grouping based on fields referenced by `@id`. Adjust the fields below based on the available columns in the loaded dataframes.

In [ ]:
# Example: Apply EDA to the first available record set
if not record_set_ids:
    print("No available record set for EDA.")
else:
    record_set_id = record_set_ids[0]  # Use the @id of the first record set
    df = dataframes[record_set_id]

    # Inspect columns/types
    print("Available columns:", df.columns.tolist())

    # Select a numeric field for analysis (find the first numeric-looking column)
    numeric_field = None
    for col in df.columns:
        if pd.api.types.is_numeric_dtype(df[col]):
            numeric_field = col
            break
    if numeric_field is None:
        # Try to coerce columns to numeric if possible
        for col in df.columns:
            try:
                df[col] = pd.to_numeric(df[col], errors='coerce')
                if df[col].notnull().sum() > 0:
                    numeric_field = col
                    break
            except Exception:
                continue

    if numeric_field is not None:
        threshold = df[numeric_field].mean() if df[numeric_field].notnull().sum() > 0 else 0
        filtered_df = df[df[numeric_field] > threshold]
        print(f"Filtered records where {numeric_field} > {threshold:.2f}:")
        display(filtered_df.head())

        # Normalize (z-score) the numeric field
        norm_field = f"{numeric_field}_normalized"
        filtered_df[norm_field] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
        print(f"\nNormalized '{numeric_field}' for filtered records:")
        display(filtered_df[[numeric_field, norm_field]].head())

        # Attempt grouping by first suitable categorical field
        group_field = None
        for col in df.columns:
            if col != numeric_field and df[col].dtype == object and df[col].nunique() > 1 and df[col].nunique() < 10:
                group_field = col
                break

        if group_field is not None:
            grouped_df = filtered_df.groupby(group_field)[numeric_field].mean()
            print(f"\nGrouped mean of '{numeric_field}' by '{group_field}':")
            display(grouped_df.head())
        else:
            print("No categorical field suitable for grouping found.")
    else:
        print("No numeric field found for EDA in the selected record set.")

## 5. Visualization

Visualize the distribution of a numeric field, and relationships between numeric and categorical fields (if present).

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Visualize numeric field distribution and group mean where possible
if record_set_ids and numeric_field is not None:
    plt.figure(figsize=(8, 4))
    sns.histplot(df[numeric_field].dropna(), kde=True)
    plt.title(f"Distribution of '{numeric_field}'")
    plt.xlabel(numeric_field)
    plt.show()

    if group_field is not None:
        plt.figure(figsize=(8, 4))
        sns.boxplot(x=df[group_field], y=df[numeric_field])
        plt.title(f"'{numeric_field}' by '{group_field}'")
        plt.xlabel(group_field)
        plt.ylabel(numeric_field)
        plt.show()
else:
    print("No numeric data available for visualization.")

## 6. Conclusion

This notebook introduced how to use `mlcroissant` for loading, exploring, and analyzing a Croissant-compliant dataset. We referenced all entities using their `@id` values and performed exploratory analysis on available record sets. The specific numeric and categorical fields, as well as their `@id`s, may change based on evolving dataset schema – review output in Section 2 for an up-to-date reference when adapting the analysis.

Key takeaways from this exploration:
* The dataset provides regression outputs and socio-demographic data on knowledge adoption in rangeland management.
* Using `mlcroissant`, it's straightforward to enumerate record sets and fields, then extract and process records by their `@id`.
* Summary statistics, normalization, filtering, grouping, and basic visualization can be easily performed in pandas once data is loaded.

For more information, visit [`mlcroissant` documentation](https://mlcroissant.ai/).